# Wersja 2: transformer BERT / HerBERT

To podejście pokazuje nowoczesny pipeline NLP:

**Tokenizer → Transformer encoder (BERT/HerBERT) → classification head**

Ta wersja wykorzystuje model wstępnie wytrenowany na dużych korpusach i następnie
**fine-tuninguje go** do zadania klasyfikacji 6 emocji.

## Domyślny model
- `allegro/herbert-base-cased`  → model BERTowy trenowany dla języka polskiego

## Alternatywa
- `google-bert/bert-base-multilingual-cased`  → wielojęzyczny BERT

# Klasyfikacja 6 emocji w języku polskim

Ten notebook pracuje na pliku `emocje_20000.csv`, który zawiera **20 000 syntetycznych tekstów po polsku** oznaczonych jedną z 6 emocji:

- `anger`
- `disgust`
- `fear`
- `joy`
- `sadness`
- `surprise`

Kolumny w pliku:
- `text`
- `label`
- `label_id`
- `split`

Notebook można uruchomić lokalnie w Jupyter Notebook / JupyterLab.

In [ ]:
# Jeśli potrzeba, odkomentuj:
# %pip install pandas numpy scikit-learn datasets transformers accelerate torch matplotlib seaborn

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

## 1. Konfiguracja

In [ ]:
CSV_PATH = "emocje_20000.csv"
MODEL_NAME = "allegro/herbert-base-cased"
# MODEL_NAME = "google-bert/bert-base-multilingual-cased"

OUTPUT_DIR = "bert_emocje_output"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 2. Wczytanie i przygotowanie danych

In [ ]:
df = pd.read_csv(CSV_PATH)

required_cols = {"text", "split"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV musi zawierać kolumny: {required_cols}")

if "label_id" not in df.columns:
    if "label" not in df.columns:
        raise ValueError("CSV musi zawierać 'label_id' lub 'label'.")
    labels = sorted(df["label"].dropna().unique().tolist())
    label_to_id = {label: idx for idx, label in enumerate(labels)}
    df["label_id"] = df["label"].map(label_to_id)
else:
    pairs = df[["label", "label_id"]].drop_duplicates().sort_values("label_id")
    label_to_id = dict(zip(pairs["label"], pairs["label_id"]))

id_to_label = {idx: label for label, idx in label_to_id.items()}

df = df.dropna(subset=["text", "label_id", "split"]).copy()
df["label_id"] = df["label_id"].astype(int)
df["text"] = df["text"].astype(str)

display(df.head())
print("Mapowanie label_to_id:", label_to_id)
print(df["label"].value_counts())

In [ ]:
train_df = df[df["split"] == "train"][["text", "label_id"]].rename(columns={"label_id": "label"})
val_df = df[df["split"] == "val"][["text", "label_id"]].rename(columns={"label_id": "label"})
test_df = df[df["split"] == "test"][["text", "label_id"]].rename(columns={"label_id": "label"})

dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    }
)

dataset

## 3. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
sample = dataset["train"][0]["text"]
encoded = tokenizer(sample)

print("Przykładowy tekst:")
print(sample)
print("\nPierwsze tokeny:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][:20]))
print("\nLiczba tokenów:", len(encoded["input_ids"]))

## 4. Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_to_id),
    id2label={int(k): v for k, v in id_to_label.items()},
    label2id=label_to_id,
)

model

## 5. Metryki

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

## 6. Konfiguracja treningu

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
)

## 7. Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 8. Trening

In [ ]:
train_result = trainer.train()
train_result

## 9. Ewaluacja na zbiorze testowym

In [ ]:
eval_metrics = trainer.evaluate(tokenized["test"])
eval_metrics

In [ ]:
predictions = trainer.predict(tokenized["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]
report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
print(report)

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
display(cm_df)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Greens")
plt.title("Confusion Matrix")
plt.ylabel("Rzeczywista klasa")
plt.xlabel("Predykcja")
plt.show()

## 10. Predykcja na nowych zdaniach

In [ ]:
examples = [
    "Jestem dziś naprawdę szczęśliwy i pełen energii.",
    "To jest obrzydliwe i nie mogę na to patrzeć.",
    "Boję się, że wszystko zaraz się zawali.",
    "Jestem wściekły na tę decyzję.",
    "Czuję ogromny smutek po tej wiadomości.",
    "Ale niespodzianka, kompletnie się tego nie spodziewałem!",
]

example_ds = Dataset.from_dict({"text": examples})
tokenized_examples = example_ds.map(tokenize, batched=True)
raw_preds = trainer.predict(tokenized_examples)
pred_ids = np.argmax(raw_preds.predictions, axis=-1)

logits = raw_preds.predictions
probs = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)

rows = []
for text, pred_id, prob_row in zip(examples, pred_ids, probs):
    rows.append({
        "text": text,
        "predicted_label": id_to_label[int(pred_id)],
        "confidence": float(prob_row[pred_id])
    })

display(pd.DataFrame(rows))

## 11. Zapis modelu i metadanych

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "best_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w", encoding="utf-8") as f:
    json.dump({"label_to_id": label_to_id, "id_to_label": id_to_label}, f, ensure_ascii=False, indent=2)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report)
    f.write("\n\nConfusion matrix:\n")
    f.write(np.array2string(cm))
    f.write("\n\nTest metrics:\n")
    f.write(json.dumps({k: float(v) for k, v in eval_metrics.items()}, ensure_ascii=False, indent=2))

print("Zapisano model i raporty w katalogu:", OUTPUT_DIR)

## 12. Co pokazać studentom jako kontrast?

### Klasyczne podejście BiLSTM
- uczy embeddingów od zera,
- analizuje sekwencję bardziej klasycznie,
- zwykle jest lżejsze i prostsze,
- jest świetne dydaktycznie.

### Nowoczesne podejście BERT / HerBERT
- startuje z dużym modelem wstępnie wytrenowanym,
- korzysta z tokenizacji subword i self-attention,
- zwykle daje lepszą jakość przy złożonym języku,
- jest bliższe współczesnym modelom językowym.